In [0]:
# ====================================================================
# SILVER LAYER - STEP 3: CREATE DIMENSION TABLES
# ====================================================================
# Purpose: Aggregate the enriched orders fact table to create
#          dimension tables for customers, products, and sellers
# ====================================================================

from pyspark.sql.functions import (
    col, count, countDistinct, sum, avg, min, max, 
    datediff, current_date, current_timestamp, round,
    first, last, when, coalesce, expr
)
from pyspark.sql.window import Window

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
SILVER_SCHEMA = f"{PROJECT_NAME}_silver"

print("=" * 80)
print("📊 SILVER LAYER - CREATE DIMENSION TABLES")
print("=" * 80)
print(f"Source: {CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")
print(f"Target: Customer, Product, and Seller Master Tables")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# LOAD ENRICHED ORDERS FACT TABLE
# ====================================================================
print("📂 Loading enriched orders fact table...\n")

enriched_orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")

print(f"✅ Loaded: {enriched_orders.count():,} rows")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# DIMENSION TABLE 1: CUSTOMERS MASTER
# ====================================================================
print("👥 Creating customers master dimension table...\n")

# Aggregate customer-level metrics from enriched orders
customers_master = (enriched_orders
    .groupBy(
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code"
    )
    .agg(
        # Order metrics
        countDistinct("order_id").alias("total_orders"),
        count("*").alias("total_items_purchased"),
        
        # Revenue metrics
        sum("item_total_value").alias("total_spent"),
        avg("item_total_value").alias("avg_item_value"),
        max("item_total_value").alias("max_item_value"),
        min("item_total_value").alias("min_item_value"),
        
        # Payment metrics
        avg("total_payment_value").alias("avg_order_payment"),
        first("primary_payment_type").alias("most_common_payment_type"),
        
        # Review metrics
        avg("review_score").alias("avg_review_score"),
        count(when(col("review_score").isNotNull(), 1)).alias("total_reviews_given"),
        
        # Delivery metrics
        avg("actual_delivery_days").alias("avg_delivery_days"),
        sum(when(col("is_late_delivery") == True, 1).otherwise(0)).alias("late_delivery_count"),
        
        # Product metrics
        countDistinct("product_category_english").alias("unique_categories_purchased"),
        countDistinct("seller_id").alias("unique_sellers_purchased_from"),
        
        # Date metrics
        min("order_purchase_timestamp").alias("first_order_date"),
        max("order_purchase_timestamp").alias("last_order_date")
    )
)

# Add calculated fields
customers_master = (customers_master
    # Calculate days since first and last order
    .withColumn(
        "days_since_first_order",
        datediff(current_date(), col("first_order_date"))
    )
    .withColumn(
        "days_since_last_order",
        datediff(current_date(), col("last_order_date"))
    )
    
    # Calculate customer lifetime (days between first and last order)
    .withColumn(
        "customer_lifetime_days",
        datediff(col("last_order_date"), col("first_order_date"))
    )
    
    # Calculate late delivery rate
    .withColumn(
        "late_delivery_rate",
        round((col("late_delivery_count") / col("total_orders")) * 100, 2)
    )
    
    # Segment customers based on recency
    .withColumn(
        "customer_segment",
        when(col("days_since_last_order") <= 90, "Active")
        .when(col("days_since_last_order") <= 180, "At Risk")
        .when(col("days_since_last_order") <= 365, "Dormant")
        .otherwise("Churned")
    )
    
    # Flag high-value customers (spent > 500 BRL)
    .withColumn(
        "is_high_value_customer",
        when(col("total_spent") >= 500, True).otherwise(False)
    )
    
    # Flag repeat customers
    .withColumn(
        "is_repeat_customer",
        when(col("total_orders") > 1, True).otherwise(False)
    )
    
    # Add audit column
    .withColumn("_created_at", current_timestamp())
)

# Write to silver schema
customers_table_name = f"{CATALOG}.{SILVER_SCHEMA}.silver_customers_master"

(customers_master.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(customers_table_name))

customer_count = customers_master.count()
print(f"✅ {customers_table_name}")
print(f"   Total customers: {customer_count:,}")
print(f"   Columns: {len(customers_master.columns)}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# DIMENSION TABLE 2: PRODUCTS MASTER
# ====================================================================
print("📦 Creating products master dimension table...\n")

# Aggregate product-level metrics from enriched orders
products_master = (enriched_orders
    .groupBy(
        "product_id",
        "product_category_name",
        "product_category_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
    .agg(
        # Sales metrics
        count("*").alias("total_units_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        
        # Revenue metrics
        sum("item_total_value").alias("total_revenue"),
        avg("price").alias("avg_price"),
        max("price").alias("max_price"),
        min("price").alias("min_price"),
        avg("freight_value").alias("avg_freight_cost"),
        
        # Review metrics
        avg("review_score").alias("avg_review_score"),
        count(when(col("review_score").isNotNull(), 1)).alias("total_reviews"),
        
        # Delivery metrics
        avg("actual_delivery_days").alias("avg_delivery_days"),
        sum(when(col("is_late_delivery") == True, 1).otherwise(0)).alias("late_delivery_count"),
        
        # Seller metrics
        countDistinct("seller_id").alias("unique_sellers"),
        
        # Date metrics
        min("order_purchase_timestamp").alias("first_sale_date"),
        max("order_purchase_timestamp").alias("last_sale_date")
    )
)

# Add calculated fields
products_master = (products_master
    # Calculate average revenue per unit
    .withColumn(
        "avg_revenue_per_unit",
        round(col("total_revenue") / col("total_units_sold"), 2)
    )
    
    # Calculate late delivery rate
    .withColumn(
        "late_delivery_rate",
        round((col("late_delivery_count") / col("total_orders")) * 100, 2)
    )
    
    # Days since last sale
    .withColumn(
        "days_since_last_sale",
        datediff(current_date(), col("last_sale_date"))
    )
    
    # Product lifecycle (days between first and last sale)
    .withColumn(
        "product_lifecycle_days",
        datediff(col("last_sale_date"), col("first_sale_date"))
    )
    
    # Calculate product volume (cm³)
    .withColumn(
        "product_volume_cm3",
        when(
            col("product_length_cm").isNotNull() & 
            col("product_height_cm").isNotNull() & 
            col("product_width_cm").isNotNull(),
            col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
        ).otherwise(None)
    )
    
    # Classify product performance
    .withColumn(
        "product_performance",
        when(col("total_units_sold") >= 100, "Best Seller")
        .when(col("total_units_sold") >= 50, "Good Seller")
        .when(col("total_units_sold") >= 20, "Average Seller")
        .otherwise("Slow Mover")
    )
    
    # Flag high-rated products
    .withColumn(
        "is_highly_rated",
        when(col("avg_review_score") >= 4.0, True).otherwise(False)
    )
    
    # Add audit column
    .withColumn("_created_at", current_timestamp())
)

# Write to silver schema
products_table_name = f"{CATALOG}.{SILVER_SCHEMA}.silver_products_master"

(products_master.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(products_table_name))

product_count = products_master.count()
print(f"✅ {products_table_name}")
print(f"   Total products: {product_count:,}")
print(f"   Columns: {len(products_master.columns)}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# DIMENSION TABLE 3: SELLERS MASTER
# ====================================================================
print("🏪 Creating sellers master dimension table...\n")

# Aggregate seller-level metrics from enriched orders
sellers_master = (enriched_orders
    .groupBy(
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_zip_code"
    )
    .agg(
        # Sales metrics
        count("*").alias("total_items_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        countDistinct("product_id").alias("unique_products_sold"),
        
        # Revenue metrics
        sum("item_total_value").alias("total_revenue"),
        avg("price").alias("avg_item_price"),
        avg("freight_value").alias("avg_freight_charged"),
        
        # Review metrics
        avg("review_score").alias("avg_review_score"),
        count(when(col("review_score").isNotNull(), 1)).alias("total_reviews_received"),
        sum(when(col("review_score") >= 4, 1).otherwise(0)).alias("positive_reviews_count"),
        sum(when(col("review_score") <= 2, 1).otherwise(0)).alias("negative_reviews_count"),
        
        # Delivery performance metrics
        avg("actual_delivery_days").alias("avg_delivery_days"),
        sum(when(col("is_late_delivery") == True, 1).otherwise(0)).alias("late_delivery_count"),
        sum(when(col("is_late_delivery") == False, 1).otherwise(0)).alias("on_time_delivery_count"),
        
        # Category metrics
        countDistinct("product_category_english").alias("unique_categories_sold"),
        
        # Date metrics
        min("order_purchase_timestamp").alias("first_sale_date"),
        max("order_purchase_timestamp").alias("last_sale_date")
    )
)

# Add calculated fields
sellers_master = (sellers_master
    # Calculate on-time delivery rate
    .withColumn(
        "on_time_delivery_rate",
        round((col("on_time_delivery_count") / col("total_orders")) * 100, 2)
    )
    
    # Calculate positive review rate
    .withColumn(
        "positive_review_rate",
        round((col("positive_reviews_count") / col("total_reviews_received")) * 100, 2)
    )
    
    # Calculate average revenue per order
    .withColumn(
        "avg_revenue_per_order",
        round(col("total_revenue") / col("total_orders"), 2)
    )
    
    # Days since last sale
    .withColumn(
        "days_since_last_sale",
        datediff(current_date(), col("last_sale_date"))
    )
    
    # Seller tenure (days between first and last sale)
    .withColumn(
        "seller_tenure_days",
        datediff(col("last_sale_date"), col("first_sale_date"))
    )
    
    # Seller performance rating
    .withColumn(
        "seller_performance_rating",
        when(
            (col("avg_review_score") >= 4.0) & (col("on_time_delivery_rate") >= 90),
            "Excellent"
        )
        .when(
            (col("avg_review_score") >= 3.5) & (col("on_time_delivery_rate") >= 80),
            "Good"
        )
        .when(
            (col("avg_review_score") >= 3.0) & (col("on_time_delivery_rate") >= 70),
            "Average"
        )
        .otherwise("Needs Improvement")
    )
    
    # Flag top sellers (revenue > 10,000 BRL)
    .withColumn(
        "is_top_seller",
        when(col("total_revenue") >= 10000, True).otherwise(False)
    )
    
    # Flag active sellers (sold in last 90 days)
    .withColumn(
        "is_active_seller",
        when(col("days_since_last_sale") <= 90, True).otherwise(False)
    )
    
    # Add audit column
    .withColumn("_created_at", current_timestamp())
)

# Write to silver schema
sellers_table_name = f"{CATALOG}.{SILVER_SCHEMA}.silver_sellers_master"

(sellers_master.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(sellers_table_name))

seller_count = sellers_master.count()
print(f"✅ {sellers_table_name}")
print(f"   Total sellers: {seller_count:,}")
print(f"   Columns: {len(sellers_master.columns)}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# DIMENSION TABLES SUMMARY
# ====================================================================
print("\n" + "=" * 80)
print("📊 DIMENSION TABLES SUMMARY")
print("=" * 80 + "\n")

dimension_tables = [
    "silver_customers_master",
    "silver_products_master",
    "silver_sellers_master"
]

print(f"{'Table Name':<40} {'Row Count':>15} {'Column Count':>15}")
print("-" * 80)

for table_name in dimension_tables:
    full_name = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    df = spark.table(full_name)
    row_count = df.count()
    col_count = len(df.columns)
    
    print(f"{table_name:<40} {row_count:>15,} {col_count:>15}")

print("=" * 80)
print("\n✅ All dimension tables created successfully!")
print("\n📝 Next Steps:")
print("  - Phase 2 (Silver Layer) is now COMPLETE")
print("  - Move to Phase 3 (Gold Layer) for business analytics")
print("  - Create gold_revenue_summary, gold_customer_cohorts, gold_rfm_scores")
print()

In [0]:
%sql
-- Show all silver tables
SHOW TABLES IN workspace.retail_silver;

In [0]:
%sql
-- Preview customers master
SELECT 
    customer_id,
    customer_city,
    customer_state,
    total_orders,
    total_spent,
    avg_review_score,
    customer_segment,
    is_repeat_customer,
    days_since_last_order
FROM workspace.retail_silver.silver_customers_master
ORDER BY total_spent DESC
LIMIT 20;

In [0]:
%sql
-- Preview products master
SELECT 
    product_id,
    product_category_english,
    total_units_sold,
    total_revenue,
    avg_review_score,
    product_performance,
    is_highly_rated
FROM workspace.retail_silver.silver_products_master
ORDER BY total_revenue DESC
LIMIT 20;

In [0]:
%sql
-- Preview sellers master
SELECT 
    seller_id,
    seller_city,
    seller_state,
    total_orders,
    total_revenue,
    avg_review_score,
    on_time_delivery_rate,
    seller_performance_rating,
    is_top_seller
FROM workspace.retail_silver.silver_sellers_master
ORDER BY total_revenue DESC
LIMIT 20;